In [0]:
BRONZE_BASE_PATH = "/Volumes/misha_azure/default/global_sales/bronze"
SILVER_BASE_PATH = "/Volumes/misha_azure/default/global_sales/silver"

In [0]:
#Imports
from pyspark.sql.functions import (
    col, to_date, when, trim, upper,
    year, month
)

In [0]:
#Read sales_transactions Bronze Table
try:
    print("Reading Bronze sales_transactions table")

    sales_bronze = spark.read.format("delta") \
        .load(f"{BRONZE_BASE_PATH}/sales_transactions")

    print("Bronze read successful")
    print("Bronze record count:", sales_bronze.count())

except Exception as e:
    print("ERROR while reading Bronze sales data")
    print("Error details:", str(e))
    raise

Reading Bronze sales_transactions table
Bronze read successful
Bronze record count: 10150


In [0]:
try:
    print("Starting Silver layer transformations")

    # Standardize data types
    sales_casted = sales_bronze \
        .withColumn("order_date", to_date(col("order_date"))) \
        .withColumn("quantity", col("quantity").cast("int")) \
        .withColumn("unit_price", col("unit_price").cast("double")) \
        .withColumn("discount", col("discount").cast("double"))

    print("Data type standardization completed")

    # Handle missing values
    sales_null_handled = sales_casted \
        .fillna({"discount": 0}) \
        .fillna({"quantity": 1})

    print("Missing values handled")

    # Standardize categorical fields
    sales_standardized = sales_null_handled \
        .withColumn("region", trim(upper(col("region")))) \
        .withColumn("sales_channel", trim(upper(col("sales_channel")))) \
        .withColumn("order_status", trim(upper(col("order_status"))))

    print("Categorical fields standardized")

    # Identify invalid records
    valid_status = ["COMPLETED", "CANCELLED"]

    invalid_sales = sales_standardized.filter(
        (col("quantity") <= 0) |
        (col("unit_price") < 0) |
        (col("order_id").isNull()) |
        (col("product_id").isNull()) |
        (~col("order_status").isin(valid_status))
    )

    print("Invalid records identified:", invalid_sales.count())

    # Valid records
    valid_sales = sales_standardized.subtract(invalid_sales)

    # Adding derived column
    sales_enriched = valid_sales \
    .withColumn(
        "revenue",
        (col("quantity") * col("unit_price")) - col("discount")
    ) \
    .withColumn("order_year", year(col("order_date"))) \
    .withColumn("order_month", month(col("order_date")))

    print("Silver enrichment completed")

except Exception as e:
    print("ERROR during Silver transformation")
    print("Error details:", str(e))
    raise

Starting Silver layer transformations
Data type standardization completed
Missing values handled
Categorical fields standardized
Invalid records identified: 537
Silver enrichment completed


In [0]:
try:
    print("Writing Silver valid sales data")

    sales_enriched.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/sales")

    print("Silver valid sales data written successfully")

    print("Writing Silver invalid sales data")

    invalid_sales.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/invalid_records/sales")

    print("Silver invalid records written successfully")

except Exception as e:
    print("ERROR while writing Silver tables")
    print("Error details:", str(e))
    raise

Writing Silver valid sales data
Silver valid sales data written successfully
Writing Silver invalid sales data
Silver invalid records written successfully


In [0]:
print("Bronze count:", sales_bronze.count())
print("Silver valid count:", sales_enriched.count())
print("Silver invalid count:", invalid_sales.count())

Bronze count: 10150
Silver valid count: 9613
Silver invalid count: 537


In [0]:
#Read Bronze products table
from pyspark.sql.functions import col, trim, upper

try:
    print("Reading Bronze PRODUCTS table")

    products_bronze = spark.read.format("delta") \
        .load(f"{BRONZE_BASE_PATH}/products")

    print("Bronze products read successful")
    print("Bronze products count:", products_bronze.count())

except Exception as e:
    print("ERROR while reading Bronze products table")
    print("Error details:", str(e))
    raise

Reading Bronze PRODUCTS table
Bronze products read successful
Bronze products count: 306


In [0]:
try:
    print("Starting Silver transformation for PRODUCTS")

    # Standardize & trim text columns
    products_std = products_bronze \
        .withColumn("category", trim(upper(col("category")))) \
        .withColumn("sub_category", trim(upper(col("sub_category")))) \
        .withColumn("brand", trim(col("brand")))

    print("Text standardization completed")

    # Remove duplicate products
    products_deduped = products_std.dropDuplicates(["product_id"])
    print("Duplicate products removed")

    # Identify invalid records
    invalid_products = products_deduped.filter(
        (col("product_id").isNull()) |
        (col("list_price") <= 0) |
        (col("cost_price") <= 0) |
        (col("cost_price") > col("list_price"))
    )

    print("Invalid products identified:", invalid_products.count())

    # Valid products
    valid_products = products_deduped.subtract(invalid_products)

    # Active / Inactive split
    inactive_products = valid_products.filter(col("active_flag") == False)
    active_products = valid_products.filter(col("active_flag") == True)

    print("Active products count:", active_products.count())
    print("Inactive products count:", inactive_products.count())

except Exception as e:
    print("ERROR during Silver PRODUCTS transformation")
    print("Error details:", str(e))
    raise

Starting Silver transformation for PRODUCTS
Text standardization completed
Duplicate products removed
Invalid products identified: 14
Active products count: 279
Inactive products count: 6


In [0]:
try:
    print("Writing Silver ACTIVE products")

    active_products.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/products")

    print("Silver active products written successfully")

    print("Writing Silver INACTIVE products")

    inactive_products.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/inactive_products")

    print("Silver inactive products written successfully")

    print("Writing Silver INVALID products")

    invalid_products.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/invalid_records/products")

    print("Silver invalid products written successfully")

except Exception as e:
    print("ERROR while writing Silver PRODUCTS tables")
    print("Error details:", str(e))
    raise

Writing Silver ACTIVE products
Silver active products written successfully
Writing Silver INACTIVE products
Silver inactive products written successfully
Writing Silver INVALID products
Silver invalid products written successfully


In [0]:
print("Bronze products:", products_bronze.count())
print("Silver valid products:", active_products.count())
print("Inactive products:", inactive_products.count())
print("Invalid products:", invalid_products.count())

Bronze products: 306
Silver valid products: 279
Inactive products: 6
Invalid products: 14


In [0]:
#Read Bronze Customer Table
try:
    print("Reading Bronze CUSTOMERS table")

    customers_bronze = spark.read.format("delta") \
        .load(f"{BRONZE_BASE_PATH}/customers")

    print("Bronze customers read successful")
    print("Bronze customers count:", customers_bronze.count())

except Exception as e:
    print("ERROR while reading Bronze customers table")
    print("Error details:", str(e))
    raise

Reading Bronze CUSTOMERS table
Bronze customers read successful
Bronze customers count: 2040


In [0]:
try:
    print("Starting Silver transformation for CUSTOMERS")

    # Standardize text columns
    customers_std = customers_bronze \
        .withColumn("customer_name", trim(col("customer_name"))) \
        .withColumn("city", trim(col("city"))) \
        .withColumn("state", trim(col("state"))) \
        .withColumn("customer_segment", trim(upper(col("customer_segment"))))

    print("Text standardization completed")

    # Remove duplicate customers
    customers_deduped = customers_std.dropDuplicates(["customer_id"])
    print("Duplicate customers removed")

    # Identify invalid customer records
    invalid_customers = customers_deduped.filter(
        (col("customer_id").isNull()) |
        (col("customer_name").isNull()) |
        (col("age") < 18) |
        (col("age") > 100)
    )

    print("Invalid customers identified:", invalid_customers.count())

    # Keep only valid customers
    valid_customers = customers_deduped.subtract(invalid_customers)

    print("Valid customers count:", valid_customers.count())

except Exception as e:
    print("ERROR during Silver CUSTOMERS transformation")
    print("Error details:", str(e))
    raise

Starting Silver transformation for CUSTOMERS
Text standardization completed
Duplicate customers removed
Invalid customers identified: 112
Valid customers count: 1888


In [0]:
try:
    print("Writing Silver VALID customers table")

    valid_customers.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/customers")

    print("Silver valid customers written successfully")

    print("Writing Silver INVALID customers table")

    invalid_customers.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/invalid_records/customers")

    print("Silver invalid customers written successfully")

except Exception as e:
    print("ERROR while writing Silver CUSTOMERS tables")
    print("Error details:", str(e))
    raise

Writing Silver VALID customers table
Silver valid customers written successfully
Writing Silver INVALID customers table
Silver invalid customers written successfully


In [0]:
print("Bronze customers:", customers_bronze.count())
print("Silver valid customers:", valid_customers.count())
print("Invalid customers:", invalid_customers.count())

Bronze customers: 2040
Silver valid customers: 1888
Invalid customers: 112


In [0]:
#Read Bronze Store Table
try:
    print("Reading Bronze STORES table")

    stores_bronze = spark.read.format("delta") \
        .load(f"{BRONZE_BASE_PATH}/stores")

    print("Bronze stores read successful")
    print("Bronze stores count:", stores_bronze.count())

except Exception as e:
    print("ERROR while reading Bronze stores table")
    print("Error details:", str(e))
    raise

Reading Bronze STORES table
Bronze stores read successful
Bronze stores count: 50


In [0]:
try:
    print("Starting Silver transformation for STORES")

    # Standardize text columns
    stores_std = stores_bronze \
        .withColumn("region", trim(upper(col("region")))) \
        .withColumn("store_type", trim(upper(col("store_type")))) \
        .withColumn("city", trim(col("city"))) \
        .withColumn("state", trim(col("state")))

    print("Text standardization completed")

    # Remove duplicate stores
    stores_deduped = stores_std.dropDuplicates(["store_id"])
    print("Duplicate stores removed")

    # Identify invalid store records
    valid_store_types = ["MALL", "STANDALONE", "WAREHOUSE"]

    invalid_stores = stores_deduped.filter(
        (col("store_id").isNull()) |
        (col("region").isNull()) |
        (~col("store_type").isin(valid_store_types))
    )

    print("Invalid stores identified:", invalid_stores.count())

    # Keep only valid stores
    valid_stores = stores_deduped.subtract(invalid_stores)

    print("Valid stores count:", valid_stores.count())

except Exception as e:
    print("ERROR during Silver STORES transformation")
    print("Error details:", str(e))
    raise

Starting Silver transformation for STORES
Text standardization completed
Duplicate stores removed
Invalid stores identified: 7
Valid stores count: 43


In [0]:
try:
    print("Writing Silver VALID stores table")

    valid_stores.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/stores")

    print("Silver valid stores written successfully")

    print("Writing Silver INVALID stores table")

    invalid_stores.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/invalid_records/stores")

    print("Silver invalid stores written successfully")

except Exception as e:
    print("ERROR while writing Silver STORES tables")
    print("Error details:", str(e))
    raise

Writing Silver VALID stores table
Silver valid stores written successfully
Writing Silver INVALID stores table
Silver invalid stores written successfully


In [0]:
print("Bronze stores:", stores_bronze.count())
print("Silver valid stores:", valid_stores.count())
print("Invalid stores:", invalid_stores.count())

Bronze stores: 50
Silver valid stores: 43
Invalid stores: 7


In [0]:
#Read Bronze Returns Table and Silver Sales Table
try:
    print("Reading Bronze RETURNS table")

    returns_bronze = spark.read.format("delta") \
        .load(f"{BRONZE_BASE_PATH}/returns")

    print("Bronze returns read successful")
    print("Bronze returns count:", returns_bronze.count())

    print("Reading Silver SALES table")

    sales_silver = spark.read.format("delta") \
        .load(f"{SILVER_BASE_PATH}/sales")

    print("Silver sales read successful")
    print("Silver sales count:", sales_silver.count())

except Exception as e:
    print("ERROR while reading Bronze Returns or Silver Sales")
    print("Error details:", str(e))
    raise

Reading Bronze RETURNS table
Bronze returns read successful
Bronze returns count: 800
Reading Silver SALES table
Silver sales read successful
Silver sales count: 9613


In [0]:
try:
    print("Starting Silver transformation for RETURNS")

    returns_std = returns_bronze \
        .withColumn("return_reason", trim(upper(col("return_reason")))) \
        .withColumn("refund_amount", col("refund_amount").cast("double"))

    print("Text standardization and casting completed")

except Exception as e:
    print("ERROR during standardization of RETURNS")
    print("Error details:", str(e))
    raise

Starting Silver transformation for RETURNS
Text standardization and casting completed


In [0]:
try:
    print("Identifying invalid RETURNS records")

    # Returns whose order_id does NOT exist in sales
    invalid_fk_returns = returns_std.join(
        sales_silver.select("order_id"),
        on="order_id",
        how="left_anti"
    )

    print("Invalid FK returns count:", invalid_fk_returns.count())

    # Numeric & null validations
    invalid_value_returns = returns_std.filter(
        (col("refund_amount") <= 0) |
        (col("order_id").isNull()) |
        (col("product_id").isNull())
    )

    print("Invalid value returns count:", invalid_value_returns.count())

    # Combine all invalid returns
    invalid_returns = invalid_fk_returns.union(invalid_value_returns).dropDuplicates()

    print("Total invalid returns identified:", invalid_returns.count())

    # Keep only valid returns
    valid_returns = returns_std.subtract(invalid_returns)

    print("Valid returns count:", valid_returns.count())

except Exception as e:
    print("ERROR during RETURNS validation logic")
    print("Error details:", str(e))
    raise

Identifying invalid RETURNS records
Invalid FK returns count: 58
Invalid value returns count: 11
Total invalid returns identified: 69
Valid returns count: 789


In [0]:
try:
    print("Writing Silver VALID returns table")

    valid_returns.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/returns")

    print("Silver valid returns written successfully")

    print("Writing Silver INVALID returns table")

    invalid_returns.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/invalid_records/returns")

    print("Silver invalid returns written successfully")

except Exception as e:
    print("ERROR while writing Silver RETURNS tables")
    print("Error details:", str(e))
    raise

Writing Silver VALID returns table
Silver valid returns written successfully
Writing Silver INVALID returns table
Silver invalid returns written successfully


In [0]:
print("Bronze returns:", returns_bronze.count())
print("Silver valid returns:", valid_returns.count())
print("Invalid returns:", invalid_returns.count())

Bronze returns: 800
Silver valid returns: 789
Invalid returns: 69


In [0]:
#Read Bronze sales_targets table
from pyspark.sql.functions import col, trim, upper

try:
    print("Reading Bronze SALES TARGETS table")

    targets_bronze = spark.read.format("delta") \
        .load(f"{BRONZE_BASE_PATH}/sales_targets")

    print("Bronze sales targets read successful")
    print("Bronze targets count:", targets_bronze.count())

except Exception as e:
    print("ERROR while reading Bronze sales targets table")
    print("Error details:", str(e))
    raise

Reading Bronze SALES TARGETS table
Bronze sales targets read successful
Bronze targets count: 48


In [0]:
try:
    print("Starting Silver transformation for SALES TARGETS")
    #Standardize
    targets_std = targets_bronze \
        .withColumn("region", trim(upper(col("region"))))

    print("Region standardization completed")

except Exception as e:
    print("ERROR during region standardization for SALES TARGETS")
    print("Error details:", str(e))
    raise


Starting Silver transformation for SALES TARGETS
Region standardization completed


In [0]:
try:
    print("Handling missing values for SALES TARGETS")

    targets_clean = targets_std.filter(
        col("region").isNotNull() &
        col("year").isNotNull() &
        col("month").isNotNull() &
        col("sales_target").isNotNull()
    )

    print("Clean sales targets count:", targets_clean.count())

except Exception as e:
    print("ERROR during missing value handling for SALES TARGETS")
    print("Error details:", str(e))
    raise

Handling missing values for SALES TARGETS
Clean sales targets count: 48


In [0]:
try:
    print("Writing Silver SALES TARGETS table")

    targets_clean.write.format("delta") \
        .mode("overwrite") \
        .save(f"{SILVER_BASE_PATH}/sales_targets")

    print("Silver sales targets written successfully")

except Exception as e:
    print("ERROR while writing Silver SALES TARGETS table")
    print("Error details:", str(e))
    raise

Writing Silver SALES TARGETS table
Silver sales targets written successfully


In [0]:
print("Bronze targets:", targets_bronze.count())
print("Silver targets:", targets_clean.count())

Bronze targets: 48
Silver targets: 48
